# Pipeline
0. Get the list of required swc files
1. Load labels parquet
2. Import the swc file
3. simplify swc file
4. attach synapse labels + neuron type
5. save simplified file
6. convert to json for find-clumpiness
7. save json file
8. calculate clumpiness for each internal node
9. attach results to the labeled swc file
10. save results.

---
# Preprocessing step
1. Create metadata labels for each swc file
2. Look for the releveant swc files only (with the wanted type)
3. Unify them via the already created function in feather file ->>> Improvement

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import polars as pl
from tqdm import tqdm
from scripts.helpers import mkdir
from joblib import Parallel, delayed
from scripts.preprocessing import simplify_swc_topology, swc2json, get_neurons_info
from scripts.processing import generate_internal_subtrees

if False:
    # Type data located in the 
    path_swc_labels = os.path.join("data", "input_labels", "neuron_data_full_article_princeton.ftr")
    swc_labels = pd.read_feather(path_swc_labels)

    required_labels = ["super_class", ["central", "optic", "visual_centrifugal", "visual_projection"]]
    swc_labels = swc_labels.loc[swc_labels[required_labels[0]].isin(required_labels[1])]

----

Objective -> attach the clumpiness score to the simplified SWC data.
1. [ ] Read the clumpiness file.
2. [ ] Get list of relevent neurons (from the parquet).
3. [ ] Get list of existing SWC files from the simplified_swc folder.
4. [ ] Define joblib for multitasking while itirating.
5. [ ] Merge simplified_swc with clumpiness.
6. [ ] Save input in another folder.

In [24]:
###########
# > Imports
import pyarrow.dataset as ds
from tqdm import tqdm
import pandas as pd
import numpy as np
import os


################
# > Define paths
path_simple1 = os.path.join("data", "input_swc", "simplified")
path_simple2 = os.path.join("data", "input_swc", "simplified_swc")
path_og = os.path.join("data", "input_swc", "sk_lod1_783_healed")
path_2process = path_simple1


############################
# > Read the clumpiness file.
# Create a dataset object (does not load data into memory yet)
dataset = ds.dataset(os.path.join("data", "unified_clumpiness.parquet"), format="parquet")


####################################################
# > Get list of relevent neurons (from the parquet).
#### EXAMPLE #### table = dataset.to_table(filter=ds.field("age") > 30, columns=["name", "age"] )  
table_neurons = dataset.to_table(columns=["neuron_id"]).to_pandas()
df_neuronsids = table_neurons.neuron_id.unique() 

# List of SWC files
swc_simplified_dir = [i.split(".")[0] for i in os.listdir(path_simple1)]
swc_og_dir = [i.split(".")[0] for i in os.listdir(path_og)]

# Nuerons to process
relv_files = np.intersect1d(df_neuronsids, swc_simplified_dir)


######################################################################
# > Itirating ove rthe SWCs files and attaching it's relevent metadata
save_path = os.path.join("data", "output_results")
for i in tqdm(relv_files):
    output_path = os.path.join(save_path, f"{i}.csv")

    if os.path.exists(output_path) is False:
        # Defining required path and loading swc + labels file
        i_path = os.path.join(path_2process, f"{i}.csv")
        i_swc = pd.read_csv(i_path, index_col=0)
        i_label = dataset.to_table(filter=(ds.field("neuron_id") == i)).to_pandas() # & 
                                          #(ds.field("property1") == "pre") & 
                                          #(ds.field("property2") == "post")).to_pandas()
        i_label.node_id = i_label.node_id.astype("int")

        # creating a unified labels column names 'label'
        i_label.insert(loc = 2, 
                    column = "label",
                    value = i_label.property1 + "_" + i_label.property2)

        # Use pivot_table instead of pivot, and specify the index
        flipped_labels =i_label.pivot_table(index=['neuron_id', 'node_id'], 
                                            columns='label',
                                            values='value',
                                            aggfunc='first').reset_index()

        # Merging SWC and labels
        i_merged = pd.merge(left=i_swc, 
                            right=flipped_labels.iloc[:,1:], 
                            left_on="node_id", 
                            right_on="node_id",
                            how="left")

        
        if os.path.exists(save_path) is False:
            os.mkdir(save_path)

        i_merged.to_csv(output_path)

    else:
        continue


100%|██████████| 20/20 [00:03<00:00,  5.57it/s]


In [28]:
import pyarrow.dataset as ds
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
from joblib import Parallel, delayed

################
# > Define paths
path_simple1 = os.path.join("data", "input_swc", "simplified")
path_simple2 = os.path.join("data", "input_swc", "simplified_swc")
path_og = os.path.join("data", "input_swc", "sk_lod1_783_healed")
path_2process = path_simple1
parquet_path = os.path.join("data", "unified_clumpiness.parquet")
save_path = os.path.join("data", "output_results")

# Ensure the output directory exists before spawning workers
if not os.path.exists(save_path):
    os.makedirs(save_path)

############################
# > Get list of relevant neurons (from the parquet)
# Read the clumpiness file metadata once in the main process
dataset = ds.dataset(parquet_path, format="parquet")
table_neurons = dataset.to_table(columns=["neuron_id"]).to_pandas()
df_neuronsids = table_neurons['neuron_id'].unique() 

# List of SWC files
swc_simplified_dir = [i.split(".")[0] for i in os.listdir(path_simple1)]
swc_og_dir = [i.split(".")[0] for i in os.listdir(path_og)]

# Neurons to process
relv_files = np.intersect1d(df_neuronsids, swc_simplified_dir)

#################################################################################################################
# > Worker function for joblib that joins SWCs file with their appropriate clumpiness results by neuron_id label.
def join_swc_clumpiness(neuron_id: str , 
                        path_2process: str, 
                        save_path: str, 
                        parquet_path: str):

    """
    neuron_id: str -> neuron id as string file.
    path_2process: str -> path to the target swc files (to which the labels will be joined).
    save_path: str -> path to the save output folder.
    parquet_path -> path to the labels parquet file.
    
    """
    i = neuron_id
    output_path = os.path.join(save_path, f"{i}.csv")

    # Early exit if the file already exists
    if os.path.exists(output_path):
        return

    # Re-initialize the pyarrow dataset inside the worker
    # This prevents pickling/serialization errors across different CPU cores
    worker_dataset = ds.dataset(parquet_path, format="parquet")

    # Defining required path and loading swc file
    i_path = os.path.join(path_2process, f"{i}.csv")
    i_swc = pd.read_csv(i_path, index_col=0)
    
    # Filter dataset for the specific neuron_id
    i_label = worker_dataset.to_table(filter=(ds.field("neuron_id") == i)).to_pandas()
    
    # Optional safety check in case a neuron ID has no corresponding labels
    if i_label.empty:
        return
        
    i_label['node_id'] = i_label['node_id'].astype("int")

    # Creating a unified labels column named 'label'
    i_label.insert(loc=2, 
                   column="label",
                   value=i_label['property1'] + "_" + i_label['property2'])

    # Use pivot_table instead of pivot, and specify the index
    flipped_labels = i_label.pivot_table(index=['neuron_id', 'node_id'], 
                                         columns='label',
                                         values='value',
                                         aggfunc='first').reset_index()

    # Merging SWC and labels
    i_merged = pd.merge(left=i_swc, 
                        right=flipped_labels.iloc[:, 1:], 
                        left_on="node_id", 
                        right_on="node_id",
                        how="left")

    # Write out the file
    i_merged.to_csv(output_path)

######################################################################
# > Iterating over the SWC files in parallel
# n_jobs=-1 tells joblib to use all available CPU cores
_capture = Parallel(n_jobs=-1)(delayed(join_swc_clumpiness)(i, path_2process, save_path, parquet_path) 
                               for i in tqdm(relv_files, desc="Dispatching Tasks"))

Dispatching Tasks: 100%|██████████| 20/20 [00:00<00:00, 12434.94it/s]
